## Loss comparison — multi-dataset (2026-03-25 collated)

This notebook compares AutoCast **collated** eval outputs for runs named
`diff_*` (flow-matching / diffusion-style processors) versus `crps_*`
(deterministic / FNO-style CRPS training), across **multiple processor
architectures** (for example FM ViT, FNO, U-Net, Swin, Azula ViT).

Default collated directory (overridable in the next cell):

- `../outputs/2026-03-25_collated/`

Pick datasets with `SELECTED_DATASETS` and pick **which processor families**
to compare with `SELECTED_MODEL_FAMILIES` (at least two keys when set). Small
versus large is inferred from parameter counts **per**
`(dataset, loss, architecture)`.

**Metric areas**

- Accuracy: VRMSE and any extra scalars in `evaluation_metrics.csv` (overall + rollout windows)
- Coverage: overall + windows + calibration curves (missing files skipped)
- Efficiency: training time, latency/throughput, params, GFLOPs
- Lead time: `rollout_metrics_per_timestep_channel_*.csv` (channel chosen in config)

Older collations without extra metric columns simply leave gaps (NaN) in plots
that aggregate those columns.


In [ ]:
from __future__ import annotations
import math
import re
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
# Config: collated results root.
# You can set an absolute path, or a relative path from your current kernel cwd.
OUTPUTS_DIR = "outputs/2026-03-25_collated"
def resolve_results_root(outputs_dir: str) -> Path:  # noqa: D103
    candidate_strings = [
        outputs_dir,
        "outputs/2026-03-25_collated",
        "../outputs/2026-03-25_collated",
        # "outputs/2026-03-16_collated",
        # "../outputs/2026-03-16_collated",
    ]
    seen = set()
    for c in candidate_strings:
        if c in seen:
            continue
        seen.add(c)
        p = Path(c).expanduser()
        p = (Path.cwd() / p).resolve() if not p.is_absolute() else p.resolve()
        if p.exists() and p.is_dir():
            return p
    # Fallback: search upward for repo-like root containing outputs.
    for parent in [Path.cwd(), *Path.cwd().parents]:
        # for leaf in ("2026-03-25_collated", "2026-03-16_collated"):
        for leaf in ("2026-03-25_collated",):
            p = (parent / "outputs" / leaf).resolve()
            if p.exists() and p.is_dir():
                return p
    # Last resort keeps diagnostics explicit.
    p = Path(outputs_dir).expanduser()
    return (Path.cwd() / p).resolve() if not p.is_absolute() else p.resolve()
RESULTS_ROOT = resolve_results_root(OUTPUTS_DIR)
# Config: dataset modules to include.
# - None or [] => use all discovered datasets.
# - Otherwise provide module names, e.g. ["ad64", "cns64", "sw2d64"].
# SELECTED_DATASETS: list[str] | None = None
SELECTED_DATASETS: list[str] | None = [
    "rd64",
    "gs64",
    "ad64",
    "sw2d64",
    "lb128x32",
    "cns64",
    "gpe_ri_high_complexity",
]
# Config: processor architecture keys to keep (Hydra processor name embedded in run folder names).
# - None or [] => use every architecture discovered after parsing run names.
# - Otherwise provide >= 2 keys (longest-prefix match in the run-name processor segment).
# Example (explicit families): ["fno_concat", "unet_small_concat", "unet_large_concat", "unet_azula_small", "unet_azula_large", "swin_small", "swin_large", "vit_azula_small", "vit_azula_large"]
# SELECTED_MODEL_FAMILIES: list[str] | None = None

# # comp 1
# SELECTED_MODEL_FAMILIES = [
#     "fno_concat",
#     # "flow_matching_vit",
#     # "unet_small_concat",
#     # "unet_large_concat",
#     "vit_azula_small",
#     "vit_azula_large",
#     "unet_azula_small",
#     "unet_azula_large",
# ]

# # comp 2
# SELECTED_MODEL_FAMILIES = [
#     "fno_concat",
#     # "flow_matching_vit",
#     # "unet_small_concat",
#     # "unet_large_concat",
#     "vit_azula_small",
#     "vit_azula_large",
#     "swin_small",
#     "swin_large",
# ]

# comp 3
SELECTED_MODEL_FAMILIES = [
    "fno_concat",
    # "flow_matching_vit",
    "unet_small_concat",
    "unet_large_concat",
    "unet_azula_small",
    "unet_azula_large",
    # "vit_azula_small",
    # "vit_azula_large",
    # "swin_small",
    # "swin_large",
]


# Longest-match token list for the processor segment (extend via EXTRA_ARCHITECTURE_PREFIXES).
ARCHITECTURE_PREFIXES: tuple[str, ...] = (
    # Legacy / existing
    "flow_matching_vit",
    "flow_matching_large",
    "flow_matching",
    "fno_concat",
    "fno",
    "vit_latent",
    "vit_large",
    "vit",
    "diffusion_vit",
    "diffusion",
    "unet_large",
    "unet",
    "swin",
    # New 2026-03-25 families (keep explicit so they do not collapse)
    "unet_azula_large",
    "unet_azula_small",
    "unet_large_concat",
    "unet_small_concat",
    "swin_large",
    "swin_small",
    "vit_azula_large",
    "vit_azula_small",
)
EXTRA_ARCHITECTURE_PREFIXES: tuple[str, ...] = ()
# Optional: stable tab20 hue order (first architecture uses tab20[0,1], next uses [2,3], ...).
# If None, order follows SELECTED_MODEL_FAMILIES, else sorted arch_key from df.
ARCHITECTURE_COLOR_ORDER: tuple[str, ...] | None = None

# If True, the plotting helper cell shows a small swatch figure proving tab20 pairing.
SHOW_ARCHITECTURE_COLOR_PREVIEW: bool = True
PLOT_EXTRA_METRICS_BARS: bool = False  # overall_* / rollout extras beyond VRMSE+coverage


# Optional subsets for heavier plot sections (None/[] => keep current df selection).
COVERAGE_DATASET_SUBSET: list[str] | None = None  # dataset_module names
COVERAGE_MODEL_FAMILY_SUBSET: list[str] | None = None  # arch_key names
LEADTIME_DATASET_SUBSET: list[str] | None = None  # dataset_module names
LEADTIME_MODEL_FAMILY_SUBSET: list[str] | None = None  # arch_key names

COVERAGE_MODEL_FAMILY_SUBSET = SELECTED_MODEL_FAMILIES
LEADTIME_MODEL_FAMILY_SUBSET = SELECTED_MODEL_FAMILIES

# Optional legend / table labels (keys are arch_key strings).
MODEL_FAMILY_DISPLAY_LABELS: dict[str, str] = {
    "flow_matching_vit": "FM ViT",
    "fno_concat": "FNO",
    "flow_matching": "FM U-Net",
    "flow_matching_large": "FM U-Net (large)",
    "vit_large": "ViT (large)",
    "vit": "ViT (small)",
    "vit_latent": "ViT latent",
    "diffusion": "Diffusion U-Net",
    "diffusion_vit": "Diffusion ViT",
    # New 2026-03-25 families
    "unet_azula_large": "Azula U-Net (large)",
    "unet_azula_small": "Azula U-Net (small)",
    "unet_large_concat": "U-Net (large)",
    "unet_small_concat": "U-Net (small)",
    "swin_large": "Swin (large)",
    "swin_small": "Swin (small)",
    "vit_azula_large": "Azula ViT (large)",
    "vit_azula_small": "Azula ViT (small)",
}
# Config: how to decide "small" vs "large".
# - "params_model_total": includes encoder+processor+decoder
# - "params_processor_total": processor-only (often the more apples-to-apples comparison)
# MODEL_SCALE_PARAM_COL = "params_model_total"
MODEL_SCALE_PARAM_COL = "params_processor_total"
# Config: per-timestep metrics channel.
# - int (0,1,2,...) -> rollout_metrics_per_timestep_channel_<int>.csv
# - "all" -> rollout_metrics_per_timestep_channel_all.csv
CHANNEL_SELECTION: int | str = 0
METRICS_PER_TIMESTEP_FILE_TEMPLATE = "rollout_metrics_per_timestep_channel_{channel}.csv"
# Optional friendly labels for module names.
DATASET_LABEL_OVERRIDES = {
    "ad64": "AD64",
    "adm32": "ADM32",
    "cns64": "CNS64",
    "gpe_ri_high_complexity": "GPE-RI high",
    "gpe_ri_low_complexity": "GPE-RI low",
    "gpehc64": "GPEHC64",
    "gpelc64": "GPELC64",
    "gs64": "GS64",
    "lb128x32": "LB128x32",
    "rd64": "RD64",
    "sw2d464": "SW2D464",
    "sw2d64": "SW2D64",
}
ROLL_WINDOWS = ["0-1", "0-4", "6-12", "13-30", "31-99"]
def dataset_label_from_module(dataset_module: str) -> str:
    if dataset_module in DATASET_LABEL_OVERRIDES:
        return DATASET_LABEL_OVERRIDES[dataset_module]
    return dataset_module.replace("_", " ").title()
def normalize_channel_token(channel_selection: int | str) -> str:
    if isinstance(channel_selection, str) and channel_selection.lower() == "all":
        return "all"
    if isinstance(channel_selection, str) and channel_selection.isdigit():
        return channel_selection
    if isinstance(channel_selection, int):
        return str(channel_selection)
    raise ValueError(f"Unsupported CHANNEL_SELECTION={channel_selection!r}; use int or 'all'.")
CHANNEL_TOKEN = normalize_channel_token(CHANNEL_SELECTION)
PER_TIMESTEP_FILENAME = METRICS_PER_TIMESTEP_FILE_TEMPLATE.format(channel=CHANNEL_TOKEN)
# Config: table display rounding.
# - None => no rounding
# - int (e.g. 2) => round numeric columns to that many significant figures for display
# DISPLAY_SIGFIGS: int | None = None
DISPLAY_SIGFIGS: int | None = 2
def round_numeric_for_display(frame: pd.DataFrame, sigfigs: int | None) -> pd.DataFrame:
    if sigfigs is None:
        return frame
    out = frame.copy()
    num_cols = out.select_dtypes(include=["number"]).columns
    def _round(v: object) -> object:
        if v is None or (isinstance(v, float) and (math.isnan(v) or math.isinf(v))):
            return v
        try:
            fv = float(v)
        except Exception:
            return v
        return float(f"{fv:.{sigfigs}g}")
    for c in num_cols:
        out[c] = out[c].map(_round)
    return out
if not RESULTS_ROOT.exists():
    print(f"Warning: RESULTS_ROOT does not exist: {RESULTS_ROOT}")
pd.set_option("display.max_columns", 300)
plt.rcParams.update({"figure.dpi": 120})


In [ ]:
# Direct loader for collated outputs.
# This avoids RunCollator edge-cases with symlinked collated directories.
_SKIP_EVAL_ROW = {"window", "batch_idx"}
_SKIP_ROLL_ROW = {"window", "batch_idx"}
def load_single_run_metrics(run_dir: Path) -> dict:
    row: dict[str, object] = {
        "run_name": run_dir.name,
        "run_path": run_dir.name,
        "dataset": None,
    }
    eval_dir = run_dir / "eval"
    # evaluation_metrics.csv -> overall_* metrics (all numeric columns on the aggregate row)
    p_eval = eval_dir / "evaluation_metrics.csv"
    if p_eval.exists():
        em = pd.read_csv(p_eval)
        if not em.empty and "window" in em.columns:
            em = em.copy()
            em["window"] = em["window"].astype(str)
            if "batch_idx" in em.columns:
                em["batch_idx"] = em["batch_idx"].astype(str)
                overall = em[(em["window"] == "all") & (em["batch_idx"] == "all")]
                if overall.empty:
                    overall = em[em["window"] == "all"]
            else:
                overall = em[em["window"] == "all"]
            if not overall.empty:
                rr = overall.iloc[0]
                for col in em.columns:
                    if col in _SKIP_EVAL_ROW:
                        continue
                    v = rr[col]
                    if pd.isna(v):
                        row[f"overall_{col}"] = np.nan
                        continue
                    num = pd.to_numeric(v, errors="coerce")
                    row[f"overall_{col}"] = float(num) if pd.notna(num) else v
    # rollout_metrics.csv -> metric_<window> columns
    p_roll = eval_dir / "rollout_metrics.csv"
    if p_roll.exists():
        rm = pd.read_csv(p_roll)
        if not rm.empty and "window" in rm.columns:
            rm = rm.copy()
            rm["window"] = rm["window"].astype(str)
            if "batch_idx" in rm.columns:
                rm = rm[rm["batch_idx"].astype(str) == "all"]
            for _, rr in rm.iterrows():
                w = rr["window"]
                for col in rm.columns:
                    if col in _SKIP_ROLL_ROW:
                        continue
                    row[f"{col}_{w}"] = pd.to_numeric(rr[col], errors="coerce")
    # benchmark_metrics.csv -> model_* and rollout_* columns
    p_bench = eval_dir / "benchmark_metrics.csv"
    if p_bench.exists():
        bm = pd.read_csv(p_bench)
        if not bm.empty and {"benchmark_type", "metric", "value"}.issubset(bm.columns):
            for _, rr in bm.iterrows():
                prefix = str(rr["benchmark_type"])
                metric = str(rr["metric"])
                row[f"{prefix}_{metric}"] = pd.to_numeric(rr["value"], errors="coerce")
    # evaluation_metadata.csv -> train_* and params_*
    p_meta = eval_dir / "evaluation_metadata.csv"
    if p_meta.exists():
        md = pd.read_csv(p_meta)
        if not md.empty and {"category", "metric", "value"}.issubset(md.columns):
            md = md.copy()
            md["category"] = md["category"].astype(str)
            md["metric"] = md["metric"].astype(str)
            rt = md[md["category"] == "runtime_train"]
            for metric_name, out_col in [
                ("mean_epoch_s", "train_mean_epoch_s"),
                ("total_s", "train_total_s"),
                ("min_epoch_s", "train_min_epoch_s"),
                ("max_epoch_s", "train_max_epoch_s"),
            ]:
                s = rt.loc[rt["metric"] == metric_name, "value"]
                if not s.empty:
                    row[out_col] = pd.to_numeric(s.iloc[0], errors="coerce")
            params = md[md["category"] == "params"]
            for metric_name, out_col in [
                ("model_total", "params_model_total"),
                ("model_trainable", "params_model_trainable"),
                ("processor_total", "params_processor_total"),
                ("encoder_total", "params_encoder_total"),
                ("decoder_total", "params_decoder_total"),
            ]:
                s = params.loc[params["metric"] == metric_name, "value"]
                if not s.empty:
                    row[out_col] = pd.to_numeric(s.iloc[0], errors="coerce")
    return row
run_dirs = sorted(
    [p for p in RESULTS_ROOT.iterdir() if p.is_dir() and (p / "eval" / "evaluation_metrics.csv").exists()],
    key=lambda p: p.name,
)
rows = [load_single_run_metrics(run_dir) for run_dir in run_dirs]
df_metrics = pd.DataFrame(rows)
# Keep object for downstream compatibility (fallbacks check this name).
metadata_by_run = {r["run_path"]: {} for r in rows if r.get("run_path")}
print(f"Discovered run dirs: {len(run_dirs)}")
print(f"Loaded metrics rows: {len(df_metrics)}")
print(f"Metadata runs: {len(metadata_by_run)}")
print(f"Results root: {RESULTS_ROOT}")
print(f"Results root exists: {RESULTS_ROOT.exists()}")
if df_metrics.empty:
    print("Warning: no metrics rows loaded. Check RESULTS_ROOT and eval CSV availability.")
df_metrics.head()


In [ ]:
def _dataset_candidates() -> list[str]:
    base = set(DATASET_LABEL_OVERRIDES.keys())
    sel = globals().get("SELECTED_DATASETS") or []
    base |= {str(x) for x in sel}
    return sorted(base, key=len, reverse=True)
def _merged_arch_prefixes() -> tuple[str, ...]:
    extra = tuple(globals().get("EXTRA_ARCHITECTURE_PREFIXES", ()))
    base = tuple(globals().get("ARCHITECTURE_PREFIXES", ()))
    merged = list(dict.fromkeys([*extra, *base]))
    return tuple(sorted(merged, key=len, reverse=True))
def arch_key_from_processor_segment(segment: str) -> str:
    seg = str(segment)
    if not seg:
        return "unknown"
    for pref in _merged_arch_prefixes():
        if seg == pref or seg.startswith(pref + "_"):
            return pref
    return seg.split("_")[0]
def parse_loss_dataset_arch(run_name: str | None) -> tuple[str | None, str | None, str | None]:
    """Return (loss_family, dataset_module, arch_processor_segment)."""
    if not run_name:
        return None, None, None
    s = str(run_name)
    m = re.match(r"^(diff|crps)_(.+)_([0-9a-f]{7}|[a-z]+)_[0-9a-f]{7}$", s)
    if not m:
        return None, None, None
    loss, mid = m.group(1), m.group(2)
    for ds in _dataset_candidates():
        pfx = ds + "_"
        if mid.startswith(pfx):
            return loss, ds, mid[len(pfx) :]
    return loss, None, mid
def normalize_dataset_module(dataset: str | None) -> str | None:
    if dataset is None or (isinstance(dataset, float) and pd.isna(dataset)):
        return None
    s = str(dataset)
    m = re.match(r"^(?P<base>.+)_(?P<suffix>[0-9a-f]{6,}|\d{6,})$", s)
    return m.group("base") if m else s
df = df_metrics.copy()
df["dataset_raw"] = df.get("dataset")
if df.empty and len(df.columns) == 0:
    raise ValueError(
        "RunCollator returned an empty metrics table (no columns). "
        f"Check RESULTS_ROOT and available runs under: {RESULTS_ROOT}"
    )
# Some collations may not expose `run_name` directly.
run_name_col = next((c for c in ["run_name", "name", "run", "run_id"] if c in df.columns), None)
if run_name_col is None and "run_path" in df.columns:
    df["run_name"] = df["run_path"].astype(str).map(lambda s: Path(s).name)
elif run_name_col is not None and run_name_col != "run_name":
    df["run_name"] = df[run_name_col].astype(str)
# If we still don't have run_name, fall back to metadata_by_run.
if "run_name" not in df.columns:
    mb = globals().get("metadata_by_run")
    if isinstance(mb, dict) and mb:
        try:
            from autocast.scripts.utils import flatten_metadata_by_run
            md_wide_fallback = flatten_metadata_by_run(mb)
            if "run_path" in md_wide_fallback.columns:
                df = md_wide_fallback.copy()
                df["dataset_raw"] = df.get("dataset", df.get("data_module", pd.NA))
                if "run_name" not in df.columns:
                    df["run_name"] = df["run_path"].astype(str).map(lambda s: Path(s).name)
            else:
                msg = "flatten_metadata_by_run output missing 'run_path' column"
                raise ValueError(msg)
        except Exception as e:
            msg = (
                "Could not infer run identifiers from collated outputs or "
                f" metadata_by_run. Available df columns: "
                f"{sorted(df.columns.tolist())}. "
                f"Fallback error: {e}"
            )
            raise ValueError(msg) from e
if "run_name" not in df.columns:
    raise ValueError(
        "Missing run identifier column even after metadata_by_run fallback. "
        f"Available columns: {sorted(df.columns.tolist())}"
    )
# Ensure run_path exists for downstream merges and CSV loading.
if "run_path" not in df.columns:
    run_path_from_name = None
    mb = globals().get("metadata_by_run")
    if isinstance(mb, dict) and mb:
        by_name = {Path(str(k)).name: str(k) for k in mb.keys()}
        run_path_from_name = df["run_name"].astype(str).map(by_name)
    if run_path_from_name is not None:
        df["run_path"] = run_path_from_name.where(run_path_from_name.notna(), df["run_name"].astype(str))
    else:
        df["run_path"] = df["run_name"].astype(str)
_parsed = df["run_name"].map(parse_loss_dataset_arch)
df["loss_family"] = _parsed.map(lambda x: x[0] if isinstance(x, tuple) else None)
df["dataset_module"] = _parsed.map(lambda x: x[1] if isinstance(x, tuple) else None)
df["arch_segment"] = _parsed.map(lambda x: x[2] if isinstance(x, tuple) else None)
# Fallback to dataset column for any run where dataset module couldn't be parsed from run_name.
fallback_dataset = df["dataset_raw"].map(normalize_dataset_module)
df["dataset_module"] = df["dataset_module"].where(df["dataset_module"].notna(), fallback_dataset)
df["arch_key"] = df["arch_segment"].map(arch_key_from_processor_segment)
df = df[df["loss_family"].isin(["diff", "crps"])].copy()
discovered_datasets = sorted(df["dataset_module"].dropna().astype(str).unique().tolist())
if not discovered_datasets:
    raise ValueError("No dataset modules discovered from collated outputs.")
if not SELECTED_DATASETS:
    selected_datasets = discovered_datasets
else:
    selected_datasets = [d for d in SELECTED_DATASETS if d in discovered_datasets]
    missing_requested = sorted(set(SELECTED_DATASETS) - set(selected_datasets))
    if missing_requested:
        print(f"Warning: requested datasets not discovered: {missing_requested}")
if not selected_datasets:
    raise ValueError("No datasets selected after filtering.")
df = df[df["dataset_module"].isin(selected_datasets)].copy()
discovered_arch_keys = sorted(df["arch_key"].dropna().astype(str).unique().tolist())
selected_model_families = globals().get("SELECTED_MODEL_FAMILIES")
if selected_model_families:
    if len(selected_model_families) < 2:
        raise ValueError("SELECTED_MODEL_FAMILIES must contain at least two keys when set.")
    df = df[df["arch_key"].isin(selected_model_families)].copy()
    missing_arch = sorted(set(selected_model_families) - set(df["arch_key"].unique()))
    if missing_arch:
        print(f"Warning: requested architectures not present after filtering: {missing_arch}")
else:
    if len(discovered_arch_keys) < 2:
        print(
            f"Warning: only one architecture key discovered ({discovered_arch_keys!r}); "
            "comparison plots need multiple groups."
        )
if df.empty:
    raise ValueError(
        "No runs left after architecture filter; check SELECTED_MODEL_FAMILIES vs keys printed above."
    )
df["dataset_family"] = df["dataset_module"]
df["dataset_label"] = df["dataset_module"].map(dataset_label_from_module)
def assign_model_scale(df_in: pd.DataFrame) -> pd.Series:
    param_col = globals().get("MODEL_SCALE_PARAM_COL", "params_model_total")
    if param_col not in df_in.columns:
        print(f"Warning: {param_col} not in df; falling back to params_model_total")
        param_col = "params_model_total"
    if param_col not in df_in.columns:
        return pd.Series("large", index=df_in.index)
    out = pd.Series("large", index=df_in.index, dtype="object")
    params = pd.to_numeric(df_in[param_col], errors="coerce")
    group_cols = [c for c in ["dataset_module", "loss_family", "arch_key"] if c in df_in.columns]
    if len(group_cols) < 2:
        return out
    for _, g in df_in.groupby(group_cols, dropna=False):
        idx = g.index
        p = params.loc[idx]
        vals = sorted(p.dropna().unique().tolist())
        if len(vals) <= 1:
            out.loc[idx] = "large"
            continue
        low, high = vals[0], vals[-1]
        out.loc[p[p == low].index] = "small"
        out.loc[p[p == high].index] = "large"
        mid_idx = p[(p != low) & (p != high)].index
        for ii in mid_idx:
            val = p.loc[ii]
            if pd.isna(val):
                out.loc[ii] = "large"
            else:
                out.loc[ii] = "small" if abs(val - low) <= abs(val - high) else "large"
    return out
df["model_scale"] = assign_model_scale(df)
df["plot_group"] = df["arch_key"].astype(str) + "__" + df["loss_family"].astype(str) + "__" + df["model_scale"].astype(str)
# Back-compat for any cell that still reads family_variant
df["family_variant"] = df["plot_group"]
# Dynamic allowlist used by downstream plotting helpers.
DATASET_ALLOWLIST = {d: dataset_label_from_module(d) for d in selected_datasets}
print(f"Discovered datasets ({len(discovered_datasets)}): {discovered_datasets}")
print(f"Selected datasets ({len(selected_datasets)}): {selected_datasets}")
print(f"Architecture keys in use: {sorted(df['arch_key'].dropna().unique().tolist())}")
print(f"Per-timestep file: {PER_TIMESTEP_FILENAME}")
print("Runs by dataset / loss / arch:")
display(
    df.groupby(["dataset_module", "loss_family", "arch_key"]).size().unstack(fill_value=0).sort_index()
)
print("Runs by plot_group:")
display(df.groupby(["dataset_module", "plot_group"]).size().unstack(fill_value=0).sort_index())
_cols = ["run_name", "loss_family", "arch_key", "dataset_raw", "dataset_module", "dataset_label"]
_cols = [c for c in _cols if c in df.columns]
_view = df[_cols].copy() if _cols else df.copy()
_sort = [c for c in ["dataset_module", "arch_key", "loss_family", "run_name"] if c in _view.columns]
if _sort:
    _view = _view.sort_values(_sort)
_view.head(40)


In [ ]:
# Metadata columns are already loaded directly in cell 2 from eval CSVs.
# Keep this cell as a light diagnostics check.
cols_meta = sorted(
    [c for c in df.columns if any(k in c for k in ["train_", "params_", "model_", "rollout_"])]
)
print(f"Metadata/perf columns found: {len(cols_meta)}")
cols_meta[:60], df.shape


## Tables

The cells below build:

- a **per-run table** for the selected dataset modules
- a **grouped summary** (mean/std/count) by `(dataset_module, loss_family)`
- **diff - crps deltas** per dataset module


In [ ]:
def existing_cols(cols: list[str]) -> list[str]:
    return [c for c in cols if c in df.columns]
def rollout_window_metric_cols(df_in: pd.DataFrame) -> list[str]:
    suffixes = tuple(f"_{w}" for w in ROLL_WINDOWS)
    out = []
    for c in df_in.columns:
        if isinstance(c, str) and c.endswith(suffixes):
            out.append(c)
    return sorted(set(out))
def overall_metric_cols(df_in: pd.DataFrame) -> list[str]:
    return sorted([c for c in df_in.columns if isinstance(c, str) and c.startswith("overall_")])
metric_cols = (
    overall_metric_cols(df)
    + [c for c in rollout_window_metric_cols(df) if c.startswith(("vrmse_", "coverage_"))]
)
perf_cols = existing_cols(metric_cols)
speed_cols = existing_cols(
    [
        "train_mean_epoch_s",
        "train_total_s",
        "params_model_total",
        "params_model_trainable",
        "model_latency_ms_per_sample",
        "model_latency_ms_per_batch",
        "model_throughput_samples_per_sec",
        "model_gflops_per_sample",
        "model_peak_gpu_memory_mb",
        "rollout_latency_ms_per_step",
        "rollout_latency_ms_per_sample",
        "rollout_throughput_samples_per_sec",
        "rollout_peak_gpu_memory_mb",
    ]
)
id_cols = existing_cols(
    [
        "run_name",
        "run_path",
        "dataset_label",
        "dataset_family",
        "arch_key",
        "loss_family",
        "model_scale",
        "plot_group",
        "processor_model",
        "processor_hidden_channels",
    ]
)
per_run = df[id_cols + perf_cols + speed_cols].copy()
sort_cols = [c for c in ["dataset_family", "arch_key", "loss_family", "model_scale", "run_name"] if c in per_run.columns]
if sort_cols:
    per_run = per_run.sort_values(sort_cols)
round_numeric_for_display(per_run, DISPLAY_SIGFIGS)


In [ ]:
group_keys = [c for c in ["dataset_family", "dataset_label", "arch_key", "loss_family"] if c in df.columns]
_core_summary = [
    "overall_vrmse",
    "overall_coverage",
    "train_mean_epoch_s",
    "model_latency_ms_per_sample",
    "model_throughput_samples_per_sec",
    "rollout_latency_ms_per_step",
    "rollout_throughput_samples_per_sec",
    "params_model_total",
    "model_gflops_per_sample",
]
_roll_pref = [f"vrmse_{w}" for w in ROLL_WINDOWS] + [f"coverage_{w}" for w in ROLL_WINDOWS]
_overall_dyn = [c for c in overall_metric_cols(df) if c not in _core_summary]
_roll_dyn = [c for c in rollout_window_metric_cols(df) if c not in _roll_pref]
summary_cols = existing_cols(_core_summary + _roll_pref + _overall_dyn + _roll_dyn)
summary = (
    df.groupby(group_keys)[summary_cols]
    .agg(["mean", "std", "count"])
    .sort_index()
)
round_numeric_for_display(summary, DISPLAY_SIGFIGS)


In [ ]:
# Diff - CRPS deltas (per dataset), split by architecture key.
primary = existing_cols(
    [
        "overall_vrmse",
        "overall_coverage",
        "train_mean_epoch_s",
        "model_latency_ms_per_sample",
        "model_throughput_samples_per_sec",
        "params_model_total",
        "model_gflops_per_sample",
    ]
)
delta_frames = []
if "arch_key" in df.columns:
    for ak, g in df.groupby("arch_key", dropna=False):
        means = g.groupby(["dataset_family", "loss_family"])[primary].mean(numeric_only=True)
        wide = means.unstack("loss_family")
        col_families = set(wide.columns.get_level_values(1)) if isinstance(wide.columns, pd.MultiIndex) else set()
        if {"diff", "crps"}.issubset(col_families):
            try:
                dpart = wide.xs("diff", axis=1, level=1) - wide.xs("crps", axis=1, level=1)
                dpart = dpart.rename(columns={c: f"diff_minus_crps__{c}" for c in dpart.columns})
                dpart = dpart.reset_index()
                dpart.insert(0, "arch_key", ak)
                delta_frames.append(dpart)
            except KeyError as e:
                print(f"Could not compute diff-crps deltas for arch_key={ak!r}: {e}")
if delta_frames:
    deltas = pd.concat(delta_frames, ignore_index=True)
    deltas = deltas.merge(
        df[["dataset_family", "dataset_label"]].drop_duplicates(), on="dataset_family", how="left"
    )
    deltas = deltas.set_index(["arch_key", "dataset_family", "dataset_label"]).sort_index()
else:
    print("No diff/crps paired means available for delta table (per architecture).")
    deltas = pd.DataFrame()
round_numeric_for_display(deltas, DISPLAY_SIGFIGS)


## Plots

Bar and line plots group runs by **`plot_group`** =
`{architecture}__{diff|crps}__{small|large}`. Missing metric values show as gaps.


In [ ]:
def dataset_labels_in_order(df_in: pd.DataFrame) -> list[str]:
    if "dataset_module" in df_in.columns:
        order_df = (
            df_in[["dataset_module", "dataset_label"]]
            .dropna()
            .drop_duplicates()
        )
        label_by_module = dict(zip(order_df["dataset_module"], order_df["dataset_label"]))
        selected = globals().get("selected_datasets")
        if selected is None:
            selected = globals().get("SELECTED_DATASETS")
        if selected:
            ordered_modules = [m for m in selected if m in label_by_module]
            extras = sorted([m for m in label_by_module.keys() if m not in set(ordered_modules)])
            ordered_modules.extend(extras)
            return [label_by_module[m] for m in ordered_modules]
        return [label_by_module[m] for m in sorted(label_by_module.keys())]
    return list(df_in["dataset_label"].dropna().drop_duplicates())




def apply_dataset_model_subsets(
    df_in: pd.DataFrame,
    dataset_subset: list[str] | None = None,
    model_family_subset: list[str] | None = None,
) -> pd.DataFrame:
    """Filter by dataset_module and arch_key when columns exist."""
    out = df_in.copy()

    if dataset_subset and "dataset_module" in out.columns:
        keep = set(str(x) for x in dataset_subset)
        out = out[out["dataset_module"].astype(str).isin(keep)].copy()

    if model_family_subset and "arch_key" in out.columns:
        keep = set(str(x) for x in model_family_subset)
        out = out[out["arch_key"].astype(str).isin(keep)].copy()

    return out

def x_label_rotation(n_labels: int) -> int:
    if n_labels <= 4:
        return 0
    if n_labels <= 8:
        return 20
    return 35


def family_group_col(df_in: pd.DataFrame) -> str:
    if "plot_group" in df_in.columns:
        return "plot_group"
    if "family_variant" in df_in.columns:
        return "family_variant"
    return "loss_family"


def _parse_plot_group(pg: object) -> tuple[str, str, str]:
    parts = str(pg).split("__")
    if len(parts) >= 3:
        return parts[0], parts[1], parts[2]
    if len(parts) == 2:
        return parts[0], parts[1], "large"
    return str(pg), "unknown", "large"


def _model_display_name(arch: str) -> str:
    labels_map = globals().get("MODEL_FAMILY_DISPLAY_LABELS", {})
    return labels_map.get(arch, arch.replace("_", " "))


def plot_group_display_label(pg: str) -> str:
    arch, loss, scale = _parse_plot_group(pg)
    model_l = _model_display_name(arch)
    loss_l = {"crps": "CRPS/FNO", "diff": "FM"}.get(loss, loss)
    return f"{model_l} · {scale} · {loss_l}"


def family_groups_in_order(df_in: pd.DataFrame) -> list[str]:
    group_col = family_group_col(df_in)
    present = set(df_in[group_col].dropna().astype(str).unique().tolist())
    sel = globals().get("SELECTED_MODEL_FAMILIES")
    arch_order = list(sel) if sel else sorted({_parse_plot_group(x)[0] for x in present})

    ordered: list[str] = []
    for arch in arch_order:
        # Keep light(small) before dark(large) per architecture.
        for scale in ("small", "large"):
            for loss in ("crps", "diff"):
                token = f"{arch}__{loss}__{scale}"
                if token in present:
                    ordered.append(token)
    for g in sorted(present):
        if g not in ordered:
            ordered.append(g)
    return ordered


def family_display_label(fam: str) -> str:
    return plot_group_display_label(fam)


def _mix_with_white(color: tuple[float, float, float, float], frac: float) -> tuple[float, float, float, float]:
    r, g, b, a = color
    return (
        r + (1.0 - r) * frac,
        g + (1.0 - g) * frac,
        b + (1.0 - b) * frac,
        a,
    )


def _mix_with_black(color: tuple[float, float, float, float], frac: float) -> tuple[float, float, float, float]:
    r, g, b, a = color
    scale = max(0.0, 1.0 - frac)
    return (r * scale, g * scale, b * scale, a)


def _rgb_luminance(rgba: tuple[float, ...]) -> float:
    r, g, b = rgba[0], rgba[1], rgba[2]
    return 0.2126 * r + 0.7152 * g + 0.0722 * b


def _mix_with_white(color: tuple[float, float, float, float], frac: float) -> tuple[float, float, float, float]:
    r, g, b, a = color
    return (
        r + (1.0 - r) * frac,
        g + (1.0 - g) * frac,
        b + (1.0 - b) * frac,
        a,
    )


def _mix_with_black(color: tuple[float, float, float, float], frac: float) -> tuple[float, float, float, float]:
    r, g, b, a = color
    scale = max(0.0, 1.0 - frac)
    return (r * scale, g * scale, b * scale, a)


def _enforce_light_dark_contrast(
    light: tuple[float, float, float, float],
    dark: tuple[float, float, float, float],
    min_delta: float = 0.22,
) -> tuple[tuple[float, float, float, float], tuple[float, float, float, float]]:
    """Guarantee visible small(light)/large(dark) separation."""
    if _rgb_luminance(light) - _rgb_luminance(dark) >= min_delta:
        return light, dark

    # Force stronger separation while preserving hue identity.
    light2 = _mix_with_white(light, 0.40)
    dark2 = _mix_with_black(dark, 0.28)
    if _rgb_luminance(light2) - _rgb_luminance(dark2) < min_delta:
        light2 = _mix_with_white(light, 0.55)
        dark2 = _mix_with_black(dark, 0.40)
    return light2, dark2


def _hue_group(arch: str) -> str:
    """Map arch_key like swin_small / swin_large to one hue family (swin)."""
    import re

    a = str(arch)
    # e.g. unet_large_concat / unet_small_concat -> unet_concat
    m = re.fullmatch(r"(.+)_(large|small)_(.+)", a)
    if m:
        return f"{m.group(1)}_{m.group(3)}"
    # e.g. swin_large, vit_azula_small -> swin, vit_azula
    m = re.fullmatch(r"(.+)_(large|small)", a)
    if m:
        return m.group(1)
    return a


def _size_from_arch_or_scale(arch: str, scale: str) -> str:
    """Prefer explicit size encoded in arch_key; fallback to plot-group scale."""
    import re

    a = str(arch)
    if re.fullmatch(r".+_(small|large)_.+", a):
        return "small" if "_small_" in a else "large"
    if a.endswith("_small"):
        return "small"
    if a.endswith("_large"):
        return "large"
    s = str(scale).lower()
    return "small" if s == "small" else "large"


def _tab20_light_dark(pair_index: int) -> tuple[tuple[float, float, float, float], tuple[float, float, float, float]]:
    """tab20 paired chips, normalized to visible light/dark contrast."""
    cmap = plt.get_cmap("tab20")
    pair_index = int(pair_index) % 10
    c0 = cmap(2 * pair_index)
    c1 = cmap(2 * pair_index + 1)
    if _rgb_luminance(c0) >= _rgb_luminance(c1):
        light, dark = c0, c1
    else:
        light, dark = c1, c0
    return _enforce_light_dark_contrast(light, dark)


def _ordered_arch_keys(df_in: pd.DataFrame) -> list[str]:
    """Stable arch_key order: ARCHITECTURE_COLOR_ORDER > SELECTED_MODEL_FAMILIES > sorted arch_key."""
    present: set[str] = set()
    base_df = globals().get("df")
    if isinstance(base_df, pd.DataFrame) and "arch_key" in base_df.columns:
        present = set(base_df["arch_key"].dropna().astype(str).unique().tolist())
    elif "arch_key" in df_in.columns:
        present = set(df_in["arch_key"].dropna().astype(str).unique().tolist())

    fixed = globals().get("ARCHITECTURE_COLOR_ORDER")
    if fixed:
        ordered = [a for a in fixed if a in present]
        ordered.extend(sorted(present - set(ordered)))
        return ordered

    sel = globals().get("SELECTED_MODEL_FAMILIES")
    if sel:
        ordered = [a for a in sel if a in present]
        ordered.extend(sorted(present - set(ordered)))
        return ordered

    if present:
        return sorted(present)

    groups = family_groups_in_order(df_in)
    out: list[str] = []
    for g in groups:
        arch, _, _ = _parse_plot_group(g)
        if arch not in out:
            out.append(arch)
    return out


def _hue_order_for_colors(df_in: pd.DataFrame) -> list[str]:
    """One tab20 pair per hue family (swin_small + swin_large -> same hue)."""
    out: list[str] = []
    for a in _ordered_arch_keys(df_in):
        h = _hue_group(a)
        if h not in out:
            out.append(h)
    return out


def _arch_color_map(df_in: pd.DataFrame) -> dict[str, tuple[tuple[float, float, float, float], tuple[float, float, float, float]]]:
    """Per hue family: (dark, light) for large/small (luminance-sorted tab20 pair)."""
    hues = _hue_order_for_colors(df_in)
    pair_map: dict[str, tuple[tuple[float, float, float, float], tuple[float, float, float, float]]] = {}
    for i, hue in enumerate(hues):
        light, dark = _tab20_light_dark(i)
        pair_map[hue] = (dark, light)
    return pair_map


def marker_for_plot_group(fam: object) -> str:
    _, loss, _ = _parse_plot_group(fam)
    return "^" if loss == "diff" else "o"


def linestyle_for_plot_group(fam: object) -> str:
    _, loss, _ = _parse_plot_group(fam)
    return "-" if loss == "diff" else "--"


def hatch_for_plot_group(fam: object) -> str:
    # Hatching distorts perceived fill colors; use linestyle/marker/edges instead.
    return ""


def build_family_style(df_in: pd.DataFrame) -> dict[str, dict[str, object]]:
    hue_pairs = _arch_color_map(df_in)
    styles: dict[str, dict[str, object]] = {}

    # One hue per model family (e.g. swin_small + swin_large share a tab20 pair).
    # small=light, large=dark. Loss = marker/linestyle/edges, not hue.
    for g in family_groups_in_order(df_in):
        arch, _, scale = _parse_plot_group(g)
        size = _size_from_arch_or_scale(arch, scale)
        hue = _hue_group(arch)
        dark, light = hue_pairs.get(hue, (plt.get_cmap("tab20")(0), plt.get_cmap("tab20")(1)))
        color = light if size == "small" else dark

        styles[g] = {
            "color": color,
            "label": plot_group_display_label(g),
            "marker": marker_for_plot_group(g),
            "linestyle": linestyle_for_plot_group(g),
            "hatch": hatch_for_plot_group(g),
        }
    return styles


def _unique_handles_labels(handles: list[object], labels: list[str]) -> tuple[list[object], list[str]]:
    seen: set[str] = set()
    out_h: list[object] = []
    out_l: list[str] = []
    for h, l in zip(handles, labels):
        if not l or l in seen:
            continue
        seen.add(l)
        out_h.append(h)
        out_l.append(l)
    return out_h, out_l


def _legend_ncols(n_items: int, max_cols: int = 4) -> int:
    # Single-column legend for side placement.
    return 1


def collect_handles_labels_from_axes(axes_in) -> tuple[list[object], list[str]]:
    handles: list[object] = []
    labels: list[str] = []
    for ax in np.array(axes_in).reshape(-1):
        h, l = ax.get_legend_handles_labels()
        handles.extend(h)
        labels.extend(l)
    return _unique_handles_labels(handles, labels)


def add_compact_legend_from_ax(ax, outside: bool = False) -> None:
    handles, labels = ax.get_legend_handles_labels()
    handles, labels = _unique_handles_labels(handles, labels)
    if not handles:
        return
    kwargs = {
        "fontsize": 8,
        "frameon": False,
        "ncol": _legend_ncols(len(labels)),
        "handlelength": 1.6,
        "columnspacing": 0.9,
    }
    if outside:
        ax.legend(handles, labels, loc="upper left", bbox_to_anchor=(1.01, 1.0), **kwargs)
    else:
        ax.legend(handles, labels, loc="best", **kwargs)


def add_compact_figure_legend(fig, handles: list[object], labels: list[str], *, y: float = 0.98) -> None:
    handles, labels = _unique_handles_labels(handles, labels)
    if not handles:
        return
    fig.legend(
        handles,
        labels,
        loc="upper center",
        bbox_to_anchor=(0.5, y),
        ncol=_legend_ncols(len(labels)),
        fontsize=8,
        frameon=False,
        handlelength=1.6,
        columnspacing=0.9,
    )


FAMILY_STYLE = build_family_style(df)


def plot_architecture_color_preview(df_in: pd.DataFrame | None = None) -> None:
    """Shows tab20 light/dark assignment per hue family (swin_small + swin_large share one row)."""
    from matplotlib.patches import Rectangle

    d = df_in if df_in is not None else globals().get("df")
    if not isinstance(d, pd.DataFrame) or "arch_key" not in d.columns:
        print("plot_architecture_color_preview: need df with arch_key")
        return
    pairs = _arch_color_map(d)
    if not pairs:
        return
    arch_by_hue: dict[str, str] = {}
    for a in sorted(d["arch_key"].dropna().astype(str).unique().tolist()):
        h = _hue_group(a)
        arch_by_hue.setdefault(h, a)

    fig, ax = plt.subplots(figsize=(7.0, max(1.2, 0.35 * len(pairs))))
    y = 0.0
    for hue in _hue_order_for_colors(d):
        if hue not in pairs:
            continue
        dark, light = pairs[hue]
        arch_ex = arch_by_hue.get(hue, hue)
        name = _model_display_name(arch_ex)
        ax.add_patch(Rectangle((0.0, y), 0.45, 0.28, facecolor=light, edgecolor="0.2", linewidth=0.6))
        ax.add_patch(Rectangle((0.55, y), 0.45, 0.28, facecolor=dark, edgecolor="0.2", linewidth=0.6))
        ax.text(1.05, y + 0.14, f"{name}  (small · large)", va="center", fontsize=9)
        y += 0.42
    ax.set_xlim(0, 3.2)
    ax.set_ylim(-0.05, y + 0.05)
    ax.axis("off")
    ax.set_title("Color pairs: tab20 per model family (small=light, large=dark)")
    plt.tight_layout()
    plt.show()


def grouped_bar(df_in: pd.DataFrame, metric: str, title: str, ylabel: str):
    group_col = family_group_col(df_in)
    required = ["dataset_label", group_col, metric]
    missing = [c for c in required if c not in df_in.columns]
    if missing:
        print(f"Skipping {metric}: missing columns {missing}")
        return

    d = df_in[required].copy()
    g = d.groupby(["dataset_label", group_col], dropna=False)[metric].mean().reset_index()
    datasets = dataset_labels_in_order(df_in)
    datasets = [ds for ds in datasets if ds in set(g["dataset_label"])]
    families = [f for f in family_groups_in_order(df_in) if f in set(g[group_col])]
    if not families:
        print(f"No grouped rows for {metric}")
        return

    x = np.arange(len(datasets), dtype=float)
    width = 0.8 / max(1, len(families))

    fig_width = max(8.0, 0.9 * len(datasets) + 2.0)
    fig, ax = plt.subplots(figsize=(fig_width, 3.8))
    styles = build_family_style(df_in)
    all_positive_finite: list[float] = []

    for i, fam in enumerate(families):
        vals = pd.to_numeric(
            g[g[group_col] == fam]
            .set_index("dataset_label")
            .reindex(datasets)[metric],
            errors="coerce",
        ).to_numpy(dtype=float)
        all_positive_finite.extend(float(v) for v in vals if np.isfinite(v) and float(v) > 0)
        offs = (i - (len(families) - 1) / 2.0) * width
        style = styles.get(fam, {"color": None, "label": plot_group_display_label(fam), "hatch": ""})
        _, loss, _ = _parse_plot_group(fam)
        ls = "-" if loss == "diff" else (0, (0.12, 0.12))
        try:
            ax.bar(
                x + offs,
                vals,
                width=width,
                label=style["label"],
                color=style["color"],
                edgecolor="0.25",
                linewidth=0.6,
                linestyle=ls,
            )
        except TypeError:
            ax.bar(
                x + offs,
                vals,
                width=width,
                label=style["label"],
                color=style["color"],
                edgecolor="0.25" if loss == "diff" else "0.55",
                linewidth=0.6,
            )

    if all_positive_finite and min(all_positive_finite) > 0:
        ax.set_yscale("log")
    else:
        ax.set_yscale("linear")

    ax.set_xticks(list(x))
    ax.set_xticklabels(datasets, rotation=x_label_rotation(len(datasets)), ha="right")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    add_compact_legend_from_ax(ax, outside=True)
    plt.tight_layout()
    plt.show()


if globals().get("SHOW_ARCHITECTURE_COLOR_PREVIEW", True):
    plot_architecture_color_preview(df)

grouped_bar(df, "overall_vrmse", "Overall VRMSE (lower is better)", "VRMSE")


In [ ]:
if "overall_coverage" in df.columns:
    grouped_bar(df, "overall_coverage", "Overall coverage", "Coverage")
# Rollout windows (core)
for w in ROLL_WINDOWS:
    col = f"vrmse_{w}"
    if col in df.columns:
        grouped_bar(df, col, f"Rollout VRMSE window {w}", "VRMSE")

# Extra overall / rollout metrics (can be many bars — off by default).
if globals().get("PLOT_EXTRA_METRICS_BARS", False):
    _baseline_overall = {
        "overall_mse",
        "overall_mae",
        "overall_rmse",
        "overall_vrmse",
        "overall_coverage",
    }
    _extra_overall = sorted([c for c in df.columns if c.startswith("overall_") and c not in _baseline_overall])
    for col in _extra_overall:
        series = pd.to_numeric(df[col], errors="coerce")
        if series.notna().any():
            grouped_bar(df, col, col.replace("overall_", "Overall ").replace("_", " "), "value")
    _win_suffixes = tuple(f"_{w}" for w in ROLL_WINDOWS)
    _extra_roll = sorted(
        {
            c
            for c in df.columns
            if isinstance(c, str) and c.endswith(_win_suffixes) and not c.startswith(("vrmse_", "coverage_"))
        }
    )
    for col in _extra_roll:
        if pd.to_numeric(df[col], errors="coerce").notna().any():
            w = col.rsplit("_", 1)[-1]
            grouped_bar(df, col, f"Rollout {col} (window {w})", "value")


In [ ]:
# Coverage vs VRMSE scatter (points = runs)
if {"overall_vrmse", "overall_coverage"}.issubset(df.columns):
    group_col = family_group_col(df)
    _cols = ["dataset_label", group_col, "overall_vrmse", "overall_coverage", "run_name"]
    _cols = [c for c in _cols if c in df.columns]
    d = df[_cols].copy()
    styles = build_family_style(df)
    d = d.dropna(subset=["overall_vrmse", "overall_coverage"])
    dataset_labels = dataset_labels_in_order(d)
    n = len(dataset_labels)
    ncols = min(4, max(1, n))
    nrows = int(math.ceil(n / ncols))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(4.2 * ncols, 3.2 * nrows),
        sharex=False,
        sharey=False,
    )
    axes = np.array(axes).reshape(-1)
    for i, ds_label in enumerate(dataset_labels):
        ax = axes[i]
        sub = d[d["dataset_label"] == ds_label]
        for fam in family_groups_in_order(d):
            ss = sub[sub[group_col] == fam]
            if ss.empty:
                continue
            style = styles.get(fam, {"color": None, "label": plot_group_display_label(fam)})
            marker = marker_for_plot_group(fam)
            ax.scatter(
                ss["overall_vrmse"],
                ss["overall_coverage"],
                label=style["label"],
                marker=marker,
                alpha=0.85,
                color=style["color"],
            )
        ax.set_title(ds_label)
        ax.set_xlabel("overall_vrmse")
        ax.set_ylabel("overall_coverage")
        ax.grid(alpha=0.25)
    for ax in axes[n:]:
        ax.set_visible(False)
    handles, labels = collect_handles_labels_from_axes(axes[:n]) if n else ([], [])
    add_compact_figure_legend(fig, handles, labels, y=0.98)
    plt.tight_layout(rect=(0, 0, 1, 0.93))
    plt.show()
else:
    print("Missing overall_vrmse/overall_coverage")


In [ ]:
# Timing / throughput / complexity comparisons (group means)
plot_df = df.copy()

# Keep grouping by architecture+loss+size even if this cell runs before upstream prep.
if "plot_group" not in plot_df.columns and {"arch_key", "loss_family"}.issubset(plot_df.columns):
    if "model_scale" not in plot_df.columns:
        plot_df["model_scale"] = plot_df["arch_key"].astype(str).map(
            lambda a: _size_from_arch_or_scale(str(a), "")
        )
    plot_df["plot_group"] = (
        plot_df["arch_key"].astype(str)
        + "__"
        + plot_df["loss_family"].astype(str)
        + "__"
        + plot_df["model_scale"].astype(str)
    )
if "family_variant" not in plot_df.columns and "plot_group" in plot_df.columns:
    plot_df["family_variant"] = plot_df["plot_group"]

param_metric = globals().get("MODEL_SCALE_PARAM_COL", "params_model_total")
param_title = {
    "params_model_total": "Model parameters (total)",
    "params_processor_total": "Model parameters (processor)",
}.get(param_metric, f"Model parameters ({param_metric})")

for metric, title, ylabel in [
    ("train_mean_epoch_s", "Mean epoch time (s) (lower is better)", "seconds"),
    ("model_latency_ms_per_sample", "Model latency per sample (ms) (lower is better)", "ms"),
    ("model_throughput_samples_per_sec", "Model throughput (samples/s) (higher is better)", "samples/s"),
    (param_metric, param_title, "params"),
    ("model_gflops_per_sample", "GFLOPs per sample", "GFLOPs"),
]:
    if metric in plot_df.columns:
        grouped_bar(plot_df, metric, title, ylabel)


## Training time comparison

These plots focus specifically on **training time**, using `evaluation_metadata.csv` fields when available.


In [ ]:
def first_present(df_in: pd.DataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in df_in.columns:
            return c
    return None
def resolve_training_columns(df_in: pd.DataFrame) -> tuple[pd.DataFrame, str | None, str | None]:
    out = df_in.copy()
    epoch_candidates = ["train_mean_epoch_s", "evaluation_runtime_train_mean_epoch_s"]
    total_candidates = ["train_total_s", "evaluation_runtime_train_total_s"]
    col_epoch = first_present(out, epoch_candidates)
    col_total = first_present(out, total_candidates)
    # Fallback 1: rebuild from metadata_by_run using flatten helper.
    if (col_epoch is None and col_total is None) and "metadata_by_run" in globals():
        try:
            from autocast.scripts.utils import flatten_metadata_by_run
            md_wide_retry = flatten_metadata_by_run(metadata_by_run)
            needed = [c for c in epoch_candidates + total_candidates if c in md_wide_retry.columns]
            if needed and "run_path" in out.columns:
                out = out.drop(columns=[c for c in needed if c in out.columns], errors="ignore")
                out = out.merge(md_wide_retry[["run_path"] + needed], on="run_path", how="left")
                col_epoch = first_present(out, epoch_candidates)
                col_total = first_present(out, total_candidates)
        except Exception as e:
            print(f"metadata_by_run fallback failed: {e}")
    # Fallback 2: read training runtime directly from evaluation_metadata.csv.
    if (col_epoch is None and col_total is None) and "run_path" in out.columns:
        rows = []
        base = RESULTS_ROOT
        for run_path in out["run_path"].dropna().astype(str).unique():
            meta_csv = base / run_path / "eval" / "evaluation_metadata.csv"
            if not meta_csv.exists():
                continue
            try:
                m = pd.read_csv(meta_csv)
                if not {"category", "metric", "value"}.issubset(m.columns):
                    continue
                mm = m[m["category"] == "runtime_train"]
                if mm.empty:
                    continue
                row = {"run_path": run_path}
                for metric_name, out_col in [
                    ("mean_epoch_s", "train_mean_epoch_s"),
                    ("total_s", "train_total_s"),
                ]:
                    s = mm.loc[mm["metric"] == metric_name, "value"]
                    if not s.empty:
                        row[out_col] = pd.to_numeric(s.iloc[0], errors="coerce")
                rows.append(row)
            except Exception:
                pass
        if rows:
            rt = pd.DataFrame(rows)
            out = out.drop(columns=[c for c in ["train_mean_epoch_s", "train_total_s"] if c in out.columns], errors="ignore")
            out = out.merge(rt, on="run_path", how="left")
            col_epoch = first_present(out, epoch_candidates)
            col_total = first_present(out, total_candidates)
    return out, col_epoch, col_total
df, col_epoch, col_total = resolve_training_columns(df)
print("epoch_time_col:", col_epoch)
print("total_train_time_col:", col_total)
def grouped_bar_with_error(df_in: pd.DataFrame, metric: str, title: str, ylabel: str):
    group_col = family_group_col(df_in)
    required = ["dataset_label", group_col, metric]
    missing = [c for c in required if c not in df_in.columns]
    if missing:
        print(f"Skipping {metric}: missing columns {missing}")
        return
    d = df_in[required].dropna().copy()
    if d.empty:
        print(f"No data for {metric}")
        return
    stats = (
        d.groupby(["dataset_label", group_col], dropna=False)[metric]
        .agg(["mean", "std", "count"])
        .reset_index()
    )
    datasets = dataset_labels_in_order(df_in)
    datasets = [ds for ds in datasets if ds in set(stats["dataset_label"])]
    families = [f for f in family_groups_in_order(df_in) if f in set(stats[group_col])]
    x = np.arange(len(datasets), dtype=float)
    width = 0.8 / max(1, len(families))
    fig_width = max(8.0, 0.9 * len(datasets) + 2.0)
    fig, ax = plt.subplots(figsize=(fig_width, 3.8))
    for i, fam in enumerate(families):
        sub = stats[stats[group_col] == fam].set_index("dataset_label").reindex(datasets)
        vals = pd.to_numeric(sub["mean"], errors="coerce").to_numpy(dtype=float)
        # If count==1, std is NaN; treat as 0 for errorbar.
        yerr = pd.to_numeric(sub["std"], errors="coerce").fillna(0.0).to_numpy(dtype=float)
        offs = (i - (len(families) - 1) / 2.0) * width
        style = build_family_style(df_in).get(fam, {"color": None, "label": plot_group_display_label(fam)})
        _, loss, _ = _parse_plot_group(fam)
        ls = "-" if loss == "diff" else (0, (0.12, 0.12))
        try:
            ax.bar(
                x + offs,
                vals,
                width=width,
                label=style["label"],
                yerr=yerr,
                capsize=4,
                color=style["color"],
                edgecolor="0.25",
                linewidth=0.6,
                linestyle=ls,
            )
        except TypeError:
            ax.bar(
                x + offs,
                vals,
                width=width,
                label=style["label"],
                yerr=yerr,
                capsize=4,
                color=style["color"],
                edgecolor="0.25" if loss == "diff" else "0.55",
                linewidth=0.6,
            )
    ax.set_xticks(list(x))
    ax.set_xticklabels(datasets, rotation=x_label_rotation(len(datasets)), ha="right")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    add_compact_legend_from_ax(ax, outside=True)
    plt.tight_layout()
    plt.show()
if col_epoch:
    grouped_bar_with_error(df, col_epoch, "Training time: mean epoch seconds (lower is better)", "seconds")
if col_total:
    grouped_bar_with_error(df, col_total, "Training time: total train seconds (lower is better)", "seconds")
# Per-run scatter for epoch time (useful even when n=1)
group_col = family_group_col(df)
if col_epoch and {"dataset_label", group_col}.issubset(df.columns):
    d = df[["dataset_label", group_col, col_epoch]].dropna()
    ds_order = dataset_labels_in_order(d)
    fig_width = max(8.0, 0.9 * len(ds_order) + 2.0)
    fig, ax = plt.subplots(figsize=(fig_width, 3.8))
    groups = [g for g in family_groups_in_order(d) if g in set(d[group_col])]
    for i, fam in enumerate(groups):
        ss = d[d[group_col] == fam]
        if ss.empty:
            continue
        # Jitter x by dataset index and family group for readability.
        x = ss["dataset_label"].map({ds: i for i, ds in enumerate(ds_order)}).astype(float)
        offset = (i - (len(groups) - 1) / 2.0) * 0.06
        style = build_family_style(df).get(fam, {"color": None, "label": plot_group_display_label(fam)})
        marker = marker_for_plot_group(fam)
        ax.scatter(x + offset, ss[col_epoch], label=style["label"], marker=marker, alpha=0.85, color=style["color"])
    ax.set_xticks(range(len(ds_order)))
    ax.set_xticklabels(ds_order, rotation=x_label_rotation(len(ds_order)), ha="right")
    ax.set_ylabel("mean_epoch_s")
    ax.set_title("Training time per run: mean_epoch_s")
    ax.grid(axis="y", alpha=0.25)
    add_compact_legend_from_ax(ax, outside=True)
    plt.tight_layout()
    plt.show()


In [ ]:
# Total epochs comparison
# If explicit epoch count is not logged, estimate as total_s / mean_epoch_s.
def resolve_epoch_count_column(df_in: pd.DataFrame) -> tuple[pd.DataFrame, str | None, str]:
    out = df_in.copy()
    explicit_candidates = [
        "train_num_epochs",
        "evaluation_runtime_train_num_epochs",
        "evaluation_runtime_train_n_epochs",
    ]
    for c in explicit_candidates:
        if c in out.columns:
            return out, c, "explicit"
    col_epoch = first_present(out, ["train_mean_epoch_s", "evaluation_runtime_train_mean_epoch_s"])
    col_total = first_present(out, ["train_total_s", "evaluation_runtime_train_total_s"])
    if col_epoch and col_total:
        denom = pd.to_numeric(out[col_epoch], errors="coerce")
        numer = pd.to_numeric(out[col_total], errors="coerce")
        est = (numer / denom).replace([float("inf"), -float("inf")], pd.NA)
        out["train_num_epochs_est"] = est.round().astype("Float64")
        return out, "train_num_epochs_est", "estimated"
    return out, None, "missing"
df, col_epochs, epoch_mode = resolve_epoch_count_column(df)
print("epochs_col:", col_epochs, "(mode:", epoch_mode, ")")
if col_epochs:
    grouped_bar_with_error(
        df,
        col_epochs,
        (
            "Training total epochs (explicit)"
            if epoch_mode == "explicit"
            else "Training total epochs (estimated: total_s / mean_epoch_s)"
        ),
        "epochs",
    )
    # Per-run scatter for epochs
    group_col = family_group_col(df)
    if {"dataset_label", group_col}.issubset(df.columns):
        d = df[["dataset_label", group_col, col_epochs]].dropna()
        ds_order = dataset_labels_in_order(d)
        fig_width = max(8.0, 0.9 * len(ds_order) + 2.0)
        fig, ax = plt.subplots(figsize=(fig_width, 3.8))
        groups = [g for g in family_groups_in_order(d) if g in set(d[group_col])]
        for i, fam in enumerate(groups):
            ss = d[d[group_col] == fam]
            if ss.empty:
                continue
            x = ss["dataset_label"].map({ds: i for i, ds in enumerate(ds_order)}).astype(float)
            offset = (i - (len(groups) - 1) / 2.0) * 0.06
            style = build_family_style(df).get(fam, {"color": None, "label": plot_group_display_label(fam)})
            marker = marker_for_plot_group(fam)
            ax.scatter(x + offset, ss[col_epochs], label=style["label"], marker=marker, alpha=0.85, color=style["color"])
        ax.set_xticks(range(len(ds_order)))
        ax.set_xticklabels(ds_order, rotation=x_label_rotation(len(ds_order)), ha="right")
        ax.set_ylabel("epochs")
        ax.set_title("Training epochs per run")
        ax.grid(axis="y", alpha=0.25)
        add_compact_legend_from_ax(ax, outside=True)
        plt.tight_layout()
        plt.show()
else:
    print("Could not derive epoch counts from available columns.")


In [ ]:
# Coverage calibration panel:
# rows = windows (all, 0-1, 0-4, 6-12, 13-30, 31-99)
# cols = datasets


def load_window_curve(run_path: str, window_label: str) -> pd.DataFrame | None:
    if window_label == "all":
        filename = "test_coverage_window_all.csv"
    else:
        filename = f"rollout_coverage_window_{window_label}.csv"

    p = RESULTS_ROOT / run_path / "eval" / filename
    if not p.exists():
        return None

    c = pd.read_csv(p)
    if c.empty or not {"coverage_level", "observed_mean"}.issubset(c.columns):
        return None

    c = c.copy()
    c["run_path"] = run_path
    c["window"] = window_label
    return c


def build_coverage_panel_data(
    df_in: pd.DataFrame, windows: list[str]
) -> tuple[pd.DataFrame, pd.DataFrame]:
    group_col = family_group_col(df_in)

    key_cols = ["run_path", "dataset_module", "dataset_label", "loss_family"]
    for optional in ["arch_key", group_col]:
        if optional in df_in.columns and optional not in key_cols:
            key_cols.append(optional)

    missing = [c for c in ["run_path", "dataset_label", "loss_family"] if c not in key_cols]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    base = df_in[key_cols].dropna(subset=["run_path", "dataset_label", "loss_family"]).drop_duplicates()
    curves = []
    missing_rows = []

    for _, r in base.iterrows():
        for w in windows:
            c = load_window_curve(str(r["run_path"]), w)
            if c is None:
                missing_rows.append(
                    {
                        "run_path": r["run_path"],
                        "dataset_module": r.get("dataset_module"),
                        "dataset_label": r["dataset_label"],
                        "loss_family": r["loss_family"],
                        "window": w,
                        "group": r.get(group_col, r["loss_family"]),
                    }
                )
                continue

            for col in key_cols:
                c[col] = r[col]
            curves.append(c)

    curves_df = pd.concat(curves, ignore_index=True) if curves else pd.DataFrame()
    missing_df = pd.DataFrame(missing_rows)

    # Re-attach authoritative run metadata from df_in so grouping columns (e.g. plot_group)
    # cannot be missing, stale, or overwritten by CSV columns with the same name.
    if not curves_df.empty:
        meta_cols = ["run_path", "dataset_module", "dataset_label", "loss_family"]
        if "arch_key" in df_in.columns:
            meta_cols.append("arch_key")
        if group_col in df_in.columns and group_col not in meta_cols:
            meta_cols.append(group_col)
        meta = (
            df_in[[c for c in meta_cols if c in df_in.columns]]
            .dropna(subset=["run_path"])
            .drop_duplicates(subset=["run_path"], keep="first")
        )
        drop_me = [c for c in meta.columns if c != "run_path" and c in curves_df.columns]
        curves_df = curves_df.drop(columns=drop_me, errors="ignore")
        curves_df = curves_df.merge(meta, on="run_path", how="left")

    return curves_df, missing_df


WINDOW_ROWS = ["all", "0-1", "0-4", "6-12", "13-30", "31-99"]
df_cov = apply_dataset_model_subsets(
    df,
    dataset_subset=globals().get("COVERAGE_DATASET_SUBSET"),
    model_family_subset=globals().get("COVERAGE_MODEL_FAMILY_SUBSET"),
)

cur_panel, missing_cov = build_coverage_panel_data(df_cov, WINDOW_ROWS)

if cur_panel.empty:
    print("No coverage-curve files found for requested windows.")
else:
    panel_group_col = family_group_col(df_cov)
    family_style = build_family_style(df_cov)

    # Diagnostics: what groups actually have curves.
    available_groups = sorted(cur_panel[panel_group_col].dropna().astype(str).unique().tolist())
    print(f"Coverage panel groups with data ({len(available_groups)}): {available_groups}")
    if not missing_cov.empty:
        print(f"Missing coverage files rows: {len(missing_cov)} (showing first 10)")
        display(missing_cov.head(10))

    dataset_cols = [label for label in dataset_labels_in_order(df_cov) if label in set(cur_panel["dataset_label"])]

    nrows = len(WINDOW_ROWS)
    ncols = max(1, len(dataset_cols))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(4.0 * ncols, 2.3 * nrows),
        sharex=True,
        sharey=True,
    )

    # Normalize axes shape
    if nrows == 1 and ncols == 1:
        axes = [[axes]]
    elif nrows == 1:
        axes = [list(axes)]
    elif ncols == 1:
        axes = [[ax] for ax in axes]

    ordered_groups = [g for g in family_groups_in_order(df_cov) if g in set(cur_panel[panel_group_col])]

    for i, w in enumerate(WINDOW_ROWS):
        for j, ds_label in enumerate(dataset_cols):
            ax = axes[i][j]
            sub = cur_panel[(cur_panel["window"] == w) & (cur_panel["dataset_label"] == ds_label)]

            # Perfect calibration line
            ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1, color="black", alpha=0.6)

            if not sub.empty:
                groups = [g for g in ordered_groups if g in set(sub[panel_group_col])]
                for fam in groups:
                    sf = sub[sub[panel_group_col] == fam]
                    if sf.empty:
                        continue

                    style = family_style.get(
                        fam,
                        {"color": None, "label": family_display_label(fam), "linestyle": "-"},
                    )

                    # Per-run curves (faint)
                    for _, rr in sf.groupby("run_path"):
                        ax.plot(
                            rr["coverage_level"],
                            rr["observed_mean"],
                            color=style["color"],
                            linestyle=style.get("linestyle", "-"),
                            alpha=0.22,
                            linewidth=1,
                        )

                    # Family mean curve (bold)
                    mean_curve = (
                        sf.groupby("coverage_level", as_index=False)["observed_mean"]
                        .mean()
                        .sort_values("coverage_level")
                    )
                    ax.plot(
                        mean_curve["coverage_level"],
                        mean_curve["observed_mean"],
                        color=style["color"],
                        linewidth=2,
                        linestyle=style.get("linestyle", "-"),
                    )

            if i == 0:
                ax.set_title(ds_label)
            if j == 0:
                ax.set_ylabel(f"{w}\nobserved_mean")
            if i == nrows - 1:
                ax.set_xlabel("coverage_level")
            ax.grid(alpha=0.2)

    legend_handles = [
        plt.Line2D(
            [0],
            [0],
            color=family_style[g]["color"],
            linestyle=family_style[g].get("linestyle", "-"),
            linewidth=2,
            label=f"{family_style[g]['label']} (mean)",
        )
        for g in ordered_groups
        if g in family_style
    ]
    legend_handles.append(
        plt.Line2D([0], [0], color="black", linestyle="--", linewidth=1, label="Perfect calibration")
    )

    add_compact_figure_legend(fig, legend_handles, [h.get_label() for h in legend_handles], y=0.995)
    plt.suptitle("Coverage calibration panel: windows x datasets", y=0.998)
    plt.tight_layout(rect=(0, 0, 1, 0.975))
    plt.show()


## Optional: coverage curves (calibration)

These plots read the per-run curve CSVs (`test_coverage_window_all.csv` and `rollout_coverage_window_*.csv`).


In [ ]:
def load_curve_csv(run_path: str, filename: str) -> pd.DataFrame | None:
    p = RESULTS_ROOT / run_path / "eval" / filename
    if not p.exists():
        return None
    c = pd.read_csv(p)
    c = c.copy()
    c["run_path"] = run_path
    return c
def plot_coverage_curves(df_in: pd.DataFrame, curve_filename: str, title: str, max_runs_per_group: int = 6):
    # Sample a few runs per dataset/family to keep plots readable.
    df_work = apply_dataset_model_subsets(
        df_in,
        dataset_subset=globals().get("COVERAGE_DATASET_SUBSET"),
        model_family_subset=globals().get("COVERAGE_MODEL_FAMILY_SUBSET"),
    )
    group_col = family_group_col(df_work)
    d = df_work[["run_path", "dataset_label", "loss_family", group_col]].drop_duplicates().copy()
    sampled = (
        d.groupby(["dataset_label", group_col], dropna=False)
        .head(max_runs_per_group)
        .reset_index(drop=True)
    )
    curves = []
    for _, r in sampled.iterrows():
        c = load_curve_csv(r["run_path"], curve_filename)
        if c is None or c.empty:
            continue
        c["dataset_label"] = r["dataset_label"]
        c["loss_family"] = r["loss_family"]
        c[group_col] = r[group_col]
        curves.append(c)
    if not curves:
        print(f"No curves found for {curve_filename}")
        return
    cur = pd.concat(curves, ignore_index=True)
    dataset_labels = [label for label in dataset_labels_in_order(df_work) if label in set(cur["dataset_label"])]
    n = len(dataset_labels)
    ncols = min(4, max(1, n))
    nrows = int(math.ceil(n / ncols))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(4.0 * ncols, 3.0 * nrows),
        sharex=True,
        sharey=True,
    )
    axes = np.array(axes).reshape(-1)
    # Consistent style per family + figure-level legend.
    family_style = build_family_style(df)
    for i, ds_label in enumerate(dataset_labels):
        ax = axes[i]
        sub = cur[cur["dataset_label"] == ds_label]
        if sub.empty:
            ax.set_title(ds_label)
            continue
        # Perfect calibration line.
        ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1, alpha=0.6, color="black")
        groups = [g for g in family_groups_in_order(cur) if g in set(sub[group_col])]
        for fam in groups:
            ss = sub[sub[group_col] == fam]
            if ss.empty:
                continue
            style = family_style.get(fam, {"color": None, "label": family_display_label(fam), "linestyle": "-"})
            for run_path, sss in ss.groupby("run_path"):
                if {"coverage_level", "observed_mean"}.issubset(sss.columns):
                    ax.plot(
                        sss["coverage_level"],
                        sss["observed_mean"],
                        alpha=0.6,
                        color=style["color"],
                        linestyle=style.get("linestyle", "-"),
                    )
        ax.set_title(ds_label)
        ax.set_xlabel("coverage_level")
        ax.set_ylabel("observed_mean")
        ax.grid(alpha=0.25)
    for ax in axes[n:]:
        ax.set_visible(False)
    # Family-level legend at figure scope.
    legend_handles = []
    for fam in family_groups_in_order(cur):
        if fam in set(cur[group_col].dropna().unique()):
            style = family_style.get(fam, {"color": None, "label": family_display_label(fam), "linestyle": "-"})
            handle = plt.Line2D([0], [0], color=style["color"], linewidth=2, label=style["label"])
            legend_handles.append(handle)
    legend_handles.append(
        plt.Line2D([0], [0], color="black", linestyle="--", linewidth=1, label="Perfect calibration")
    )
    add_compact_figure_legend(fig, legend_handles, [h.get_label() for h in legend_handles], y=0.98)
    plt.suptitle(title)
    plt.tight_layout(rect=(0, 0, 1, 0.93))
    plt.show()
plot_coverage_curves(
    df,
    curve_filename="test_coverage_window_all.csv",
    title="Test coverage calibration curves (window=all)",
)


In [ ]:
# Example rollout window curve (adjust filename as needed)
plot_coverage_curves(
    df,
    # curve_filename="rollout_coverage_window_0-4.csv",
    curve_filename="rollout_coverage_window_6-12.csv",
    title="Rollout coverage calibration curves (window 6-12)",
)


In [ ]:
# Lead-time metrics from per-timestep rollout files
#
# This section loads eval/{PER_TIMESTEP_FILENAME} for each selected run
# and compares Diff vs CRPS as a function of lead time.
#
# Change CHANNEL_SELECTION in the config cell to switch between
# channel_0, channel_1, ... and channel_all.
# Missing per-timestep files are skipped and reported.


In [ ]:
def load_per_timestep_metrics(df_in: pd.DataFrame, filename: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    group_col = family_group_col(df_in)
    key_cols = ["run_path", "dataset_module", "dataset_label", "loss_family"]
    if "arch_key" in df_in.columns:
        key_cols.append("arch_key")
    if group_col in df_in.columns and group_col not in key_cols:
        key_cols.append(group_col)

    req = ["run_path", "dataset_label", "loss_family"]
    base = df_in[key_cols].dropna(subset=req).drop_duplicates().copy()
    rows = []
    missing_rows = []
    for r in base.itertuples(index=False):
        run_path = str(r.run_path)
        p = RESULTS_ROOT / run_path / "eval" / filename
        if not p.exists():
            missing_rows.append(
                {
                    "run_path": run_path,
                    "dataset_module": r.dataset_module,
                    "dataset_label": r.dataset_label,
                    "loss_family": r.loss_family,
                }
            )
            continue
        raw = pd.read_csv(p, index_col=0)
        if raw.empty:
            continue
        long = raw.reset_index().rename(columns={raw.index.name or "index": "metric"})
        long = long.melt(id_vars="metric", var_name="timestep", value_name="value")
        long["timestep"] = pd.to_numeric(long["timestep"], errors="coerce")
        long["value"] = pd.to_numeric(long["value"], errors="coerce")
        long = long.dropna(subset=["metric", "timestep", "value"]).copy()
        if long.empty:
            continue
        long["run_path"] = run_path
        long["dataset_module"] = r.dataset_module
        long["dataset_label"] = r.dataset_label
        long["loss_family"] = r.loss_family
        if "arch_key" in base.columns:
            long["arch_key"] = r.arch_key
        if group_col in base.columns:
            long[group_col] = getattr(r, group_col)
        rows.append(long)
    metrics_long = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    missing_df = pd.DataFrame(missing_rows)

    # Authoritative run metadata so plot_group (architecture × loss × scale) cannot be
    # missing or clobbered by CSV columns sharing the same name as df_in columns.
    if not metrics_long.empty:
        meta_cols = ["run_path", "dataset_module", "dataset_label", "loss_family"]
        if "arch_key" in df_in.columns:
            meta_cols.append("arch_key")
        if group_col in df_in.columns and group_col not in meta_cols:
            meta_cols.append(group_col)
        meta = (
            df_in[[c for c in meta_cols if c in df_in.columns]]
            .dropna(subset=["run_path"])
            .drop_duplicates(subset=["run_path"], keep="first")
        )
        drop_me = [c for c in meta.columns if c != "run_path" and c in metrics_long.columns]
        metrics_long = metrics_long.drop(columns=drop_me, errors="ignore")
        metrics_long = metrics_long.merge(meta, on="run_path", how="left")

    return metrics_long, missing_df
def plot_metric_vs_lead_time(
    metrics_long: pd.DataFrame,
    metric: str,
    dataset_labels: list[str],
    max_timestep: int | None = None,
):
    sub = metrics_long[metrics_long["metric"] == metric].copy()
    if max_timestep is not None:
        sub = sub[sub["timestep"] <= max_timestep].copy()
    if sub.empty:
        print(f"No rows for metric={metric!r}")
        return
    labels = [d for d in dataset_labels if d in set(sub["dataset_label"])]
    n = len(labels)
    ncols = min(4, max(1, n))
    nrows = int(math.ceil(n / ncols))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(4.2 * ncols, 3.0 * nrows),
        sharex=True,
        # sharey=True,
        sharey=False,
    )
    axes = np.array(axes).reshape(-1)
    group_col = family_group_col(sub)
    styles = build_family_style(metrics_long)
    for i, ds_label in enumerate(labels):
        ax = axes[i]
        ds = sub[sub["dataset_label"] == ds_label]
        groups = [g for g in family_groups_in_order(ds) if g in set(ds[group_col])]
        for fam in groups:
            sf = ds[ds[group_col] == fam]
            if sf.empty:
                continue
            agg = (
                sf.groupby("timestep", as_index=False)["value"]
                .agg(["mean", "std", "count"])
                .reset_index()
                .sort_values("timestep")
            )
            style = styles.get(fam, {"color": None, "label": family_display_label(fam)})
            ax.plot(
                agg["timestep"],
                agg["mean"],
                color=style["color"],
                linewidth=2,
                linestyle=style.get("linestyle", "-"),
                label=style["label"],
            )
            has_multi = (agg["count"] > 1).any()
            if has_multi:
                y1 = agg["mean"] - agg["std"].fillna(0.0)
                y2 = agg["mean"] + agg["std"].fillna(0.0)
                ax.fill_between(agg["timestep"], y1, y2, color=style["color"], alpha=0.15)
        ax.set_title(ds_label)
        ax.set_xlabel("lead time (timestep)")
        ax.set_ylabel(metric)
        ax.grid(alpha=0.25)
    for ax in axes[n:]:
        ax.set_visible(False)
    present_groups = [g for g in family_groups_in_order(sub) if g in set(sub[group_col])]
    handles = [
        plt.Line2D(
            [0],
            [0],
            color=styles[g]["color"],
            linewidth=2,
            linestyle=styles[g].get("linestyle", "-"),
            label=styles[g]["label"],
        )
        for g in present_groups
        if g in styles
    ]
    add_compact_figure_legend(fig, handles, [h.get_label() for h in handles], y=0.98)
    plt.suptitle(f"{metric}: lead-time comparison ({PER_TIMESTEP_FILENAME})")
    plt.tight_layout(rect=(0, 0, 1, 0.93))
    plt.show()
df_lead = apply_dataset_model_subsets(
    df,
    dataset_subset=globals().get("LEADTIME_DATASET_SUBSET"),
    model_family_subset=globals().get("LEADTIME_MODEL_FAMILY_SUBSET"),
)
metrics_long, missing_per_timestep = load_per_timestep_metrics(df_lead, PER_TIMESTEP_FILENAME)
print(f"Per-timestep rows loaded: {len(metrics_long)}")
if not missing_per_timestep.empty:
    print(
        f"Missing {PER_TIMESTEP_FILENAME} for {len(missing_per_timestep)} runs "
        f"(showing first 10):"
    )
    display(missing_per_timestep.head(10))
else:
    print(f"All selected runs include {PER_TIMESTEP_FILENAME}.")
if metrics_long.empty:
    print("No per-timestep metrics found to plot.")
else:
    available_metrics = sorted(metrics_long["metric"].dropna().unique().tolist())
    print(f"Available per-timestep metrics ({len(available_metrics)}): {available_metrics}")
    preferred_metrics = [
        "vrmse",
        "rmse",
        "mae",
        "mse",
        "coverage_0.9",
        "coverage_0.75",
        "coverage_0.5",
        "coverage_0.25",
        "coverage_0.1",
    ]
    metrics_to_plot = [m for m in preferred_metrics if m in available_metrics]
    if not metrics_to_plot:
        metrics_to_plot = available_metrics[:4]
    label_order = dataset_labels_in_order(df_lead)
    # Plot 1: same as before (small multiples, wrapping grid)
    for metric_name in metrics_to_plot:
        plot_metric_vs_lead_time(
            metrics_long,
            metric=metric_name,
            dataset_labels=label_order,
            max_timestep=None,
        )
    # Plot 2: one dataset per column (metrics as rows).
    # For many datasets this gets wide, so we chunk columns into multiple figures.
    LEADTIME_COLS_PER_FIG: int = 7
    def plot_metrics_vs_lead_time_dataset_columns(
        metrics_long: pd.DataFrame,
        metrics: list[str],
        dataset_labels: list[str],
        cols_per_fig: int = 6,
        max_timestep: int | None = None,
    ):
        if metrics_long.empty or not metrics:
            return
        group_col = family_group_col(metrics_long)
        styles = build_family_style(metrics_long)
        # Keep dataset order but only those present.
        present_ds = set(metrics_long["dataset_label"].dropna().astype(str))
        labels = [d for d in dataset_labels if d in present_ds]
        if not labels:
            return
        # Column chunking for readability.
        cols_per_fig = max(1, int(cols_per_fig))
        for start in range(0, len(labels), cols_per_fig):
            chunk = labels[start : start + cols_per_fig]
            nrows = len(metrics)
            ncols = len(chunk)
            fig, axes = plt.subplots(
                nrows,
                ncols,
                figsize=(3.6 * ncols, 2.7 * nrows),
                sharex=True,
                sharey=False,
                constrained_layout=False,
            )
            axes = np.array(axes)
            if nrows == 1:
                axes = axes.reshape(1, -1)
            if ncols == 1:
                axes = axes.reshape(-1, 1)
            for r, metric in enumerate(metrics):
                sub = metrics_long[metrics_long["metric"] == metric].copy()
                if max_timestep is not None:
                    sub = sub[sub["timestep"] <= max_timestep].copy()
                for c, ds_label in enumerate(chunk):
                    ax = axes[r, c]
                    ds = sub[sub["dataset_label"] == ds_label]
                    subplot_pos_values: list[float] = []
                    is_coverage_metric = str(metric).startswith("coverage_")
                    groups = [g for g in family_groups_in_order(ds) if g in set(ds[group_col])]
                    for fam in groups:
                        sf = ds[ds[group_col] == fam]
                        if sf.empty:
                            continue
                        agg = (
                            sf.groupby("timestep", as_index=False)["value"]
                            .agg(["mean", "std", "count"])
                            .reset_index()
                            .sort_values("timestep")
                        )
                        style = styles.get(fam, {"color": None, "label": family_display_label(fam)})
                        if is_coverage_metric:
                            mean_plot = agg["mean"].clip(lower=0.0, upper=1.0)
                        else:
                            mean_plot = agg["mean"].clip(lower=1e-6)
                            subplot_pos_values.extend(mean_plot[mean_plot > 0].tolist())
                        ax.plot(
                            agg["timestep"],
                            mean_plot,
                            color=style["color"],
                            linewidth=2,
                            linestyle=style.get("linestyle", "-"),
                        )
                        has_multi = (agg["count"] > 1).any()
                        if has_multi:
                            if is_coverage_metric:
                                y1 = (agg["mean"] - agg["std"].fillna(0.0)).clip(lower=0.0, upper=1.0)
                                y2 = (agg["mean"] + agg["std"].fillna(0.0)).clip(lower=0.0, upper=1.0)
                            else:
                                y1 = (agg["mean"] - agg["std"].fillna(0.0)).clip(lower=1e-6)
                                y2 = (agg["mean"] + agg["std"].fillna(0.0)).clip(lower=1e-6)
                                subplot_pos_values.extend(y1[y1 > 0].tolist())
                                subplot_pos_values.extend(y2[y2 > 0].tolist())
                            ax.fill_between(agg["timestep"], y1, y2, color=style["color"], alpha=0.15)
                    if r == 0:
                        ax.set_title(ds_label)
                    if r == nrows - 1:
                        ax.set_xlabel("lead time (timestep)")
                    if c == 0:
                        ax.set_ylabel(metric)
                    ax.grid(alpha=0.25)
                    is_coverage_metric = str(metric).startswith("coverage_")
                    if is_coverage_metric:
                        ax.set_yscale("linear")
                        ax.set_ylim(bottom=0.0, top=1.0)
                    else:
                        ymin = max(1e-6, min(subplot_pos_values) * 0.8) if subplot_pos_values else 1e-6
                        ymax = max(subplot_pos_values) * 1.25 if subplot_pos_values else 10.0
                        ax.set_yscale("log")
                        ax.set_ylim(bottom=ymin, top=ymax)
            present_groups = [g for g in family_groups_in_order(metrics_long) if g in set(metrics_long[group_col])]
            handles = [
                plt.Line2D(
                    [0],
                    [0],
                    color=styles[g]["color"],
                    linewidth=2,
                    linestyle=styles[g].get("linestyle", "-"),
                    label=styles[g]["label"],
                )
                for g in present_groups
                if g in styles
            ]
            add_compact_figure_legend(fig, handles, [h.get_label() for h in handles], y=0.98)
            plt.suptitle(
                f"Lead-time comparison ({PER_TIMESTEP_FILENAME}) — datasets as columns ({start+1}-{min(start+len(chunk), len(labels))}/{len(labels)})"
            )
            plt.tight_layout(rect=(0, 0, 1, 0.92))
            plt.show()
    error_metrics = [m for m in metrics_to_plot if not str(m).startswith("coverage_")]
    coverage_metrics = [m for m in metrics_to_plot if str(m).startswith("coverage_")]
    if error_metrics:
        print(f"Plotting error-metric lead-time panel for: {error_metrics}")
        plot_metrics_vs_lead_time_dataset_columns(
            metrics_long,
            metrics=error_metrics,
            dataset_labels=label_order,
            cols_per_fig=LEADTIME_COLS_PER_FIG,
            max_timestep=None,
        )
    if coverage_metrics:
        print(f"Plotting coverage-metric lead-time panel for: {coverage_metrics}")
        plot_metrics_vs_lead_time_dataset_columns(
            metrics_long,
            metrics=coverage_metrics,
            dataset_labels=label_order,
            cols_per_fig=LEADTIME_COLS_PER_FIG,
            max_timestep=None,
        )


In [ ]:
# Grouped results table: dataset -> model type -> size
# Includes error metrics, coverage metrics, and compute/timing columns when available.
import numpy as np
import pandas as pd
# Prefer precomputed merged table when available.
if "res" in globals():
    summary_df = res.copy()
# Fallback: build from df + df_lookup if possible.
elif "df" in globals():
    summary_df = df.copy()
else:
    raise ValueError(
        "No source table found. Run the collator cell first to define `df`."
    )
# --- Derive grouping columns ---
# Dataset (best-effort from available columns)
dataset_candidates = [
    "dataset_label",
    "dataset_family",
    "dataset_module",
    "Dataset",
    "dataset",
    "dataset_name",
    "datamodule",
    "datamodule.data_path",
    "Data",
]
dataset_col = next((c for c in dataset_candidates if c in summary_df.columns), None)
if dataset_col is None:
    summary_df["dataset_group"] = "unknown"
else:
    dataset_series = summary_df[dataset_col].astype(str)
    if dataset_col == "datamodule.data_path":
        dataset_series = dataset_series.map(lambda x: x.rstrip("/").split("/")[-1])
    summary_df["dataset_group"] = dataset_series
# Model label and model type (prefer existing notebook grouping columns)
model_label_col = (
    "Short name"
    if "Short name" in summary_df.columns
    else ("processor_model" if "processor_model" in summary_df.columns else "run_name")
)
summary_df["model_label"] = summary_df[model_label_col].astype(str)
if "arch_key" in summary_df.columns and "loss_family" in summary_df.columns:
    labels_map = globals().get("MODEL_FAMILY_DISPLAY_LABELS", {})
    arch_disp = summary_df["arch_key"].astype(str).map(lambda a: labels_map.get(a, a.replace("_", " ")))
    loss_disp = summary_df["loss_family"].astype(str).replace({"crps": "CRPS/FNO", "diff": "FM"})
    summary_df["model_label"] = arch_disp.astype(str) + " / " + loss_disp.astype(str)
if "loss_family" in summary_df.columns:
    # Reuse notebook-native grouping (diff/crps) when already computed upstream.
    summary_df["model_type"] = summary_df["loss_family"].astype(str).replace({"crps": "FNO", "diff": "FM"})
else:
    type_source = summary_df["model_label"].str.lower()
    if "run_name" in summary_df.columns:
        type_source = type_source.fillna("") + " " + summary_df["run_name"].astype(str).str.lower()
    summary_df["model_type"] = np.select(
        [
            type_source.str.contains("fno", na=False),
            type_source.str.contains("flow_matching|fm", na=False),
        ],
        ["FNO", "FM"],
        default="Other",
    )
# Size split (prefer model_scale created earlier in notebook)
if "model_scale" in summary_df.columns:
    size_raw = summary_df["model_scale"].astype(str).str.lower()
    summary_df["size_group"] = np.select(
        [
            size_raw.str.contains("small", na=False),
            size_raw.str.contains("large", na=False),
        ],
        ["small", "large"],
        default="unknown",
    )
else:
    if "small" in summary_df.columns:
        size_bool = summary_df["small"]
    elif "processor_lt_10" in summary_df.columns:
        size_bool = summary_df["processor_lt_10"]
    elif "Processor params" in summary_df.columns:
        params_raw = summary_df["Processor params"]
        params_num = pd.to_numeric(params_raw, errors="coerce")
        # Fallback for values like "8 channels" or "12.0 params"
        missing_num = params_num.isna()
        if missing_num.any():
            extracted = pd.to_numeric(
                params_raw.astype(str).str.extract(r"([-+]?\d*\.?\d+)")[0],
                errors="coerce",
            )
            params_num = params_num.where(~missing_num, extracted)
        size_bool = params_num < 10
    else:
        size_bool = pd.Series([pd.NA] * len(summary_df), index=summary_df.index)
    summary_df["size_group"] = np.select(
        [size_bool.eq(True), size_bool.eq(False)],
        ["small", "large"],
        default="unknown",
    )
# --- Collect metric columns ---
error_metric_candidates = [
    "overall_vrmse",
    "vrmse_0-1",
    "vrmse_6-12",
    "vrmse_13-30",
    "vrmse_31-99",
    "overall_crps",
]
coverage_metric_candidates = [
    "overall_coverage",
    "coverage_0-1",
    "coverage_6-12",
    "coverage_13-30",
    "coverage_31-99",
]
error_cols = [c for c in error_metric_candidates if c in summary_df.columns]
coverage_cols = [c for c in coverage_metric_candidates if c in summary_df.columns]
# --- Build compute/timing columns (if available) ---
compute_cols = []
# Prefer compute columns already present in df/res.
compute_candidates = [
    "train_total_s",
    "train_mean_epoch_s",
    "model_latency_ms_per_sample",
    "model_throughput_samples_per_sec",
    "rollout_latency_ms_per_sample",
    "rollout_throughput_samples_per_sec",
    "params_model_total",
    "params_model_trainable",
]
for col in compute_candidates:
    if col in summary_df.columns and col not in compute_cols:
        compute_cols.append(col)
if "Train time mins" in summary_df.columns and "train_total_s" not in summary_df.columns:
    summary_df["train_total_s"] = pd.to_numeric(summary_df["Train time mins"], errors="coerce") * 60.0
    compute_cols.append("train_total_s")
if "metadata" in globals() and isinstance(metadata, dict) and len(metadata) > 0:
    meta_all = pd.concat(metadata.values(), ignore_index=True).copy()
    meta_all["value"] = pd.to_numeric(meta_all["value"], errors="coerce")
    def _loader_metric(loader_name: str, metric_name: str, out_col: str) -> pd.DataFrame:
        out = meta_all[
            (meta_all["loader"] == loader_name) & (meta_all["metric"] == metric_name)
        ][["run_name", "value"]].copy()
        out = out.rename(columns={"value": out_col})
        return out.groupby("run_name", as_index=False)[out_col].mean()
    infer_pb = _loader_metric("test_dataloader", "per_batch_mean_s", "inference_per_batch_s")
    test_total = _loader_metric("test_dataloader", "total_s", "test_total_s")
    rollout_total = _loader_metric("rollout_dataloader", "total_s", "rollout_total_s")
    for frame in (infer_pb, test_total, rollout_total):
        summary_df = summary_df.merge(frame, on="run_name", how="left")
    for col in ["inference_per_batch_s", "test_total_s", "rollout_total_s"]:
        if col in summary_df.columns:
            compute_cols.append(col)
all_value_cols = error_cols + coverage_cols + compute_cols
if not all_value_cols:
    raise ValueError("No error/coverage/compute columns found for summary table.")
# Optional: focus on FNO only (as requested model type). Set to False to keep all types.
FNO_ONLY = False
if FNO_ONLY:
    summary_df = summary_df[summary_df["model_type"] == "FNO"].copy()
# --- Aggregate table ---
# Add derived compute columns with canonical names if available.
def _first_present(candidates: list[str], cols: list[str]) -> str | None:
    for c in candidates:
        if c in cols:
            return c
    return None
if "params_model_total" in summary_df.columns:
    summary_df["params_m"] = pd.to_numeric(summary_df["params_model_total"], errors="coerce") / 1e6
if "model_latency_ms_per_sample" in summary_df.columns:
    summary_df["latency_per_sample_ms"] = pd.to_numeric(
        summary_df["model_latency_ms_per_sample"], errors="coerce"
    )
if "model_gflops_per_sample" in summary_df.columns:
    summary_df["gflops_per_sample"] = pd.to_numeric(
        summary_df["model_gflops_per_sample"], errors="coerce"
    )
# Aggregate with derived columns included.
agg_map = {col: "mean" for col in all_value_cols}
agg_map["run_name"] = "nunique"
for extra_col in [
    "train_mean_epoch_s",
    "latency_per_sample_ms",
    "params_m",
    "gflops_per_sample",
]:
    if extra_col in summary_df.columns:
        agg_map[extra_col] = "mean"
group_cols = [c for c in ["dataset_group", "arch_key", "model_type", "size_group"] if c in summary_df.columns]
summary_table = (
    summary_df
    .groupby(group_cols, dropna=False)
    .agg(agg_map)
    .rename(columns={"run_name": "n_runs"})
    .sort_index()
)
# Requested metric subset (with robust fallback names).
selected_cols = [
    # "n_runs",
    _first_present(["overall_vrmse"], summary_table.columns.tolist()),
    _first_present(["vrmse_0-1"], summary_table.columns.tolist()),
    _first_present(["vrmse_6-12"], summary_table.columns.tolist()),
    _first_present(["vrmse_13-30"], summary_table.columns.tolist()),
    _first_present(["vrmse_31-99"], summary_table.columns.tolist()),
    _first_present(["overall_coverage"], summary_table.columns.tolist()),
    _first_present(["coverage_0-1"], summary_table.columns.tolist()),
    _first_present(["coverage_6-12"], summary_table.columns.tolist()),
    _first_present(["coverage_13-30"], summary_table.columns.tolist()),
    _first_present(["coverage_31-99"], summary_table.columns.tolist()),
    _first_present(["train_mean_epoch_s"], summary_table.columns.tolist()),
    _first_present(["latency_per_sample_ms", "model_latency_ms_per_sample"], summary_table.columns.tolist()),
    _first_present(["params_m"], summary_table.columns.tolist()),
    _first_present(["gflops_per_sample", "model_gflops_per_sample"], summary_table.columns.tolist()),
]
selected_cols = [c for c in selected_cols if c is not None]
summary_subset = summary_table[selected_cols].copy()
# Rename to concise report labels.
rename_map = {
    "latency_per_sample_ms": "latency_per_sample",
    "model_latency_ms_per_sample": "latency_per_sample",
    "params_m": "params_millions",
    "gflops_per_sample": "gflops",
    "model_gflops_per_sample": "gflops",
}
summary_subset = summary_subset.rename(columns=rename_map)
# Scientific notation for compact markdown/display.
def _fmt_sci(x: object) -> object:
    if isinstance(x, (int, np.integer)):
        return int(x)
    if isinstance(x, (float, np.floating)):
        if np.isnan(x):
            return ""
        return f"{x:.2e}"
    return x
summary_md = summary_subset.reset_index().copy()
for col in summary_md.columns:
    if col == "n_runs":
        continue
    summary_md[col] = summary_md[col].map(_fmt_sci)
# Markdown display for copy/paste into docs.
print(summary_md.to_markdown(index=False))
# Notebook display: force narrower wrapped columns.
styled_summary = (
    summary_subset.style
    .set_table_styles(
        [
            {
                "selector": "table",
                "props": [
                    ("table-layout", "fixed"),
                    ("width", "100%"),
                ],
            },
            {
                "selector": "th.col_heading",
                "props": [
                    ("white-space", "normal"),
                    ("max-width", "64px"),
                    ("min-width", "64px"),
                    ("overflow-wrap", "anywhere"),
                    ("word-break", "break-word"),
                    ("line-height", "1.1"),
                ],
            },
            {
                "selector": "td",
                "props": [
                    ("white-space", "normal"),
                    ("max-width", "64px"),
                    ("min-width", "64px"),
                    ("overflow-wrap", "anywhere"),
                    ("word-break", "break-word"),
                    ("line-height", "1.1"),
                ],
            },
        ]
    )
    .format(lambda v: _fmt_sci(v))
)
styled_summary